## -------------------------------------------------------------------------------------------------------------------

# PROYECTO DEL CEIEC - Network Traffic Classification, en colaboración con Foqum
# Descripción del notebook: preprocesamiento de archivos "pcap" a semiflujos etiquetados
# Becario responsable del proyecto: Carlos Arranz Luque
# Directora de la investigación: Ana María Maitín López
# Fecha: curso académico 2022-2023

## -------------------------------------------------------------------------------------------------------------------

## WARNING: 64GB DE RAM NECESARIOS SI SE QUIERE PREPROCESAR TODOS LOS ARCHIVOS PCAP, DE LAS 5 CATEGORÍAS DEL DATASET "ISCX-VPN-NonVPN-2016"

## -------------------------------------------------------------------------------------------------------------------

## Importación de librerías y módulos

In [1]:
import numpy as np
import pandas as pd
import configparser
import time
import sys
from numpy import dstack

## Módulos/scripts propios
from src import * 


## Este código recarga los módulos/librerías antes de cada ejecución parcial del código, de tal forma
## que podemos cambiar una función en mitad de la ejecución y dichos cambios serán efectivos sin necesidad
## de recargar el kernel. Esto es útil cuando aún se está en fase de desarrollo y hay que verificar si las
## funciones tienen el comportamiento deseado
%load_ext autoreload
%autoreload 2

2023-07-08 15:50:28.297945: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## -------------------------------------------------------------------------------------------------------------------

## Opciones de ejecución e hiperparámetros

In [2]:
## Leemos el archivo "config.cfg" donde están las opciones de ejecución e hiperparámetros del preprocesamiento

config = configparser.RawConfigParser()
config.read('config/config_preprocesamiento.cfg')

## Optenemos los valores de cada una de las subcategorías del archivo de configuración
opcionesEjecucion_dict = dict(config.items('OpcionesEjecucion'))
hiperparametros_dict = dict(config.items('Hiperparametros'))
rutasArchivos_dict = dict(config.items('RutasArchivos'))

In [3]:
### ----------------------------------------------------------------------------------------------
### ------------------------------ Opciones de ejecución -----------------------------------------
### ----------------------------------------------------------------------------------------------

# Si está a True, se generan los archivos CSV de los archivos PCAP. Esto lleva mucho tiempo y solo 
# es necesario hacerlo una vez, ya que los ficheros CSV se guardan en el directorio 'Archivos_CSV'
# de forma persistente 
# "eval()" nos traduce el string "True" o "False" al boleano correspondiente
conf_generar_ficheros_CSV = eval(opcionesEjecucion_dict['generar_ficheros_csv'])

# Si está a True, se generan los dataframes de los archivos CSV. Esta opción no es  necesaria si hemos
# generado estos dataframes anteriormente, ya que se guardan de forma persistente en el directorio
# 'Dataframes_preprocesados'
conf_generar_DFs = eval(opcionesEjecucion_dict['generar_dfs'])



### ----------------------------------------------------------------------------------------------
### --------------------------------- Hiperparámetros --------------------------------------------
### ----------------------------------------------------------------------------------------------

# categorias_a_procesar --> Categorías a PROCESAR por el programa, si por ejemplo "Chat" NO está en esta lista,
#                           esta categoría NO se procesará.
# min_paquetes_flujo --> Indica cuál debe ser el mínimo de paquetes de un flujo, con menos será desechado, ya que se entiende
#                         que es demasiado corto y por tanto no es relevante para poder aportar información.
# tiempo_maximo_semiflujo --> Máximo de tiempo que puede pasar entre el primer paquete de un semiflujo y el último, si no se
#                             cumple se entiende que se ha roto la relación entre dichos paquetes y no es un semiflujo válido.
# num_paquetes_semiflujo --> Número de paquetes que conformará un semiflujo, si no llega se desecha y cuando llega al tamaño
#                            indicado el semiflujo se completa, guardándolo y empezando uno nuevo con el siguiente paquete
#                            del flujo.
# longitud_raw --> Longitud fija a la que se truncará el contenido RAW del paquete, es decir, su TCPpayload o su UDPdata, se
#                  realizará padding con ceros en caso de que este no llegue a la longitud. 
# num_hex_token --> Número de caracteres hexadecimales que tendrá cada uno de los tokens en los que se dividirá el contenido
#                   raw del paquete. Con 4, los tokens serán de la forma 'XXXX', siendo X un carácter hexadecimal.
# token_separador --> Estrategia a seguir a la hora de decidir cómo es el token separador de fin de paquete.
#                     De momento solo se permite la opción de 'num_máximo', que pone el número más grande posible, como token.
#                     Para un 'num_hex_token' = 4, esto sería 'FFFF'


categorias_a_procesar = hiperparametros_dict['categorias_a_procesar'].split()

min_paquetes_flujo = int(hiperparametros_dict['min_paquetes_flujo'])

tiempo_maximo_semiflujo = float(hiperparametros_dict['tiempo_maximo_semiflujo'])

num_paquetes_semiflujo = int(hiperparametros_dict['num_paquetes_semiflujo'])

longitud_raw = int(hiperparametros_dict['longitud_raw'])

num_hex_token = int(hiperparametros_dict['num_hex_token'])

token_separador = hiperparametros_dict['token_separador']



### ----------------------------------------------------------------------------------------------
### ----------------------------------- Rutas archivos -------------------------------------------
### ----------------------------------------------------------------------------------------------

# En "pathAbsolutoPCAP" tenemos la ruta absoluta donde estan guardados los archivos PCAP, en este directorio
# debería de haber varias carpetas, cada una identificando el tipo de archivos PCAP que contiene
# En "pathRelativoCSV" tenemos la ruta relativa donde almacenaremos los CSV que vamos a generar si tenemos
# a "True" la opción correspondiente
# En "pathRelativoDFS" tenemos la ruta relativa donde almacenaremos los DataFrames que vamos a generar si tenemos
# a "True" la opción correspondiente
# En "pathRelativoSemiflujos" tenemos la ruta relativa donde almacenaremos los semiflujos, que es la salida más
# importante del programa

pathAbsolutoPCAP = rutasArchivos_dict["pathabsolutopcap"]
pathRelativoCSV = rutasArchivos_dict["pathrelativocsv"]
pathRelativoDFS = rutasArchivos_dict["pathrelativodfs"]
pathRelativoSemiflujos = rutasArchivos_dict["pathrelativosemiflujos"]



## -------------------------------------------------------------------------------------------------------------------

## Mostramos los parámetros leídos del archivo de configuración 
### (Así nos cercioramos qué ha leído el programa y evitamos errores de configuración)

In [4]:
print("Parámetros leídos del archivo de configuración: \n")
print("Generar ficheros CSV = "+str(conf_generar_ficheros_CSV))
print("Generar DFs = "+str(conf_generar_DFs))
print("Categorias a procesar = "+str(categorias_a_procesar))
print("Minimo de paquetes por flujo = "+str(min_paquetes_flujo))
print("Tiempo maximo de un semiflujo = "+str(tiempo_maximo_semiflujo))
print("Numero de paquetes por semiflujo = "+str(num_paquetes_semiflujo))
print("Longitud del contenido raw de un paquete = "+str(longitud_raw))
print("Numero tokens hexadecimales por paquete = "+str(num_hex_token))
print("Token separador = "+token_separador)
print("Path absolutoPCAP = "+pathAbsolutoPCAP)
print("Path relativo CSV = "+pathRelativoCSV)
print("Path relativo DFS = "+pathRelativoDFS)
print("Path relativo semiflujos = "+pathRelativoSemiflujos)


Parámetros leídos del archivo de configuración: 

Generar ficheros CSV = False
Generar DFs = False
Categorias a procesar = ['YouTube', 'Spotify', 'Discord']
Minimo de paquetes por flujo = 20
Tiempo maximo de un semiflujo = 30.0
Numero de paquetes por semiflujo = 20
Longitud del contenido raw de un paquete = 800
Numero tokens hexadecimales por paquete = 4
Token separador = num_maximo
Path absolutoPCAP = /home/ceiec/app/CEIEC/PROYECTO 5G CLASIFICACION/DATASETS/GrabacionesPFG/
Path relativo CSV = Objetos preprocesados/CSVs/
Path relativo DFS = Objetos preprocesados/Dataframes/
Path relativo semiflujos = Objetos preprocesados/Semiflujos/


## -------------------------------------------------------------------------------------------------------------------

## Obtenemos el hash del los hiperparámetros del preprocesamiento, será necesario para identificar y distinguir los semiflujos de este preprocesamiento de otros

In [5]:
## Este hash identifica los semiflujos que van a ser creados, ya que se forma a partir de todos sus hiperparámetros
hashSemiflujos = md5hash(str(hiperparametros_dict))
hashSemiflujos

'060eaab55e1e0dd2d41fe88ed69ebb5b'

## -------------------------------------------------------------------------------------------------------------------

## Rutas del programa

###  ------------------------------------------------------- ¡¡¡AVISO!!! ----------------------------------------------------------------------
### Esta siguiente celda cambiará en función de las distintas categorías de archivos PCAP que se utilicen para entrenar el modelo. En el caso actual (inicios de 2023), se dispone de los archivos PCAP del dataset  "ISCX-VPN-NonVPN-2016", que tiene 5 categorías --> Chat, Email, FileTransfer, Streaming, VoIP 
### [Enlace al dataset](https://www.unb.ca/cic/datasets/vpn.html)
### En el caso de querer adaptar el programa a un nuevo tipo de archivos PCAP habrá que actualizar el código con las categorías correspondientes de los nuevos archivos
###  ----------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
## "pathAbsolutoPCAP" es el path del directorio que contiene las diferentes carpetas donde se encuentran los archivos
## PCAP que vayamos a utilizar. Debemos asegurarnos que dentro de este directorio existan carpetas que coincidan con
## las categorías que hemos indicado en el archivo de configuración que íbamos a procesar

## "listaPathPCAP" contiene el listado de los directorios finales donde se encuentran las grabaciones en 
## formato PCAP. Este listado contendrá tantos paths como categorías hemos indicado que vamos a procesar

## "listaPathCSV" contiene el listado de los subdirectorios donde se van a guardar los archivos CSV generados
## a partir de los archivos PCAP correspondientes. Sigue el esquema de categorías de los archivos PCAP

listaPathPCAP = []
listaPathCSV = []

for categoria in categorias_a_procesar:
    listaPathPCAP.append(pathAbsolutoPCAP + categoria)
    

for categoria in categorias_a_procesar:
    listaPathCSV.append(pathRelativoCSV +"CSV_" +categoria)
    
    isExist = os.path.exists(listaPathCSV[-1])
    if not isExist:
        # Create a new directory because it does not exist
        os.makedirs(listaPathCSV[-1])
        print("\nDirectorio para los CSVs '"+listaPathCSV[-1]+"' creado!")


print("\nListado paths PCAP:")
print(listaPathPCAP)
print("\nListado paths CSV:")
print(listaPathCSV)


Listado paths PCAP:
['/home/ceiec/app/CEIEC/PROYECTO 5G CLASIFICACION/DATASETS/GrabacionesPFG/YouTube', '/home/ceiec/app/CEIEC/PROYECTO 5G CLASIFICACION/DATASETS/GrabacionesPFG/Spotify', '/home/ceiec/app/CEIEC/PROYECTO 5G CLASIFICACION/DATASETS/GrabacionesPFG/Discord']

Listado paths CSV:
['Objetos preprocesados/CSVs/CSV_YouTube', 'Objetos preprocesados/CSVs/CSV_Spotify', 'Objetos preprocesados/CSVs/CSV_Discord']


## -------------------------------------------------------------------------------------------------------------------

## En algunos bloques de código que son pesados en tiempo de ejecución, se seguirá la siguiente estructura, de tal forma que se podrá saber cuánto ha tardado en ejecutarse 

In [7]:
# start_time = time.time()

## ----------------------------------------------------------
## -------------- CÓDIGO A EJECUTARSE -----------------------
## ----------------------------------------------------------

# calcular_tiempo(start_time)

## -------------------------------------------------------------------------------------------------------------------

## Comienzo activo del programa

### Si la opción 'conf_generar_ficheros_CSV' está a 'True', vamos a generar los ficheros CSV de todos los archivos PCAP que se encuentran en los directorios seleccionados en el paso anterior. Si está a 'False', deberemos asegurarnos que estos ficheros CSV ya hayan sido generados en ocaciones anteriores

In [8]:
## Recorremos ambas listas a la vez, pasando a la función "generar_CSVs" el directorio donde se encuentran 
## los archivos PCAP y el subdirectorio donde tiene que guardar los archivos CSV que genere de esos PCAP


if(conf_generar_ficheros_CSV == True):
    
    ## Comprobamos que TShark esté instalado ya que es necesario para poder generar los archivos CSV
    if (is_tool("tshark") == False):
        print("\n-------------------------------ERROR---------------------------------\n")
        print("TShark no existe en el sistema, el programa no se podra ejecutar correctamente...")
        print("\n-------------------------------ERROR---------------------------------\n")
        sys.exit("ERROR TSHARK")
        
    start_time = time.time()
    
    for pathPCAP, pathCSV in zip(listaPathPCAP,listaPathCSV):       
        generar_CSVs(pathPCAP, pathCSV)

    calcular_tiempo(start_time)

### Si la opción 'conf_generar_DFs' está a 'True' generamos los dataframes, de lo contrario recuperamos los existentes en el disco

In [9]:

if(conf_generar_DFs == False):
    ## Recuperamos el listado de dataframes preprocesados
    DF_diccionario = cargar_diccionario_DFS(categorias_a_procesar,pathRelativoDFS)

## No existen dataframes preprocesados o queremos Volver a generarlos
else:
    start_time = time.time()

    ## Llamamos a la función que genera los DFS a partir de las listas de CSVS
    DF_diccionario = procesar_csvs_a_dfs(listaPathCSV,categorias_a_procesar)
    
    ## Guardamos el listado de dataframes del diccionario, en el directorio adecuado, para tenerlos ya generados
    ## para futuras ocaciones
    guardar_diccionario_DFS(DF_diccionario,pathRelativoDFS)          

    calcular_tiempo(start_time)
        
## Anotamos con cuantas clases estamos trabajando en este preprocesamiento en concreto, ya que
## será necesario saberlo más adelante
num_clases_existentes = len(DF_diccionario)

### Vamos a imprimir un DF para ver cómo se ha transformado la información desde nuestro CSV
### Va a tener solo 4 columnas: 'Time', 'TCPpayload', 'UDPpayload' y 'Hash', este último indica a que conversación/flujo pertenece ese paquete
### La columna TCPpayload estará a NaN si el paquete sigue el protocolo UDP, y vicecersa con UDPpayload si el paquete sigue el protocolo TCP

In [10]:
categoria0 = list(DF_diccionario.keys())[0]
DF_diccionario[categoria0][0]

,Time,TCPpayload,UDPpayload,Hash
0,2.336035,NaN,48f84a1d4e0b089813a5d193e496363d96f23f8f424952...,f22c891361a2bcc2ca93c23104632357
1,2.336072,NaN,48f84a1d4e0b089813a5d193e496363d96f23f8f424952...,f22c891361a2bcc2ca93c23104632357
2,2.336354,NaN,54f84a1d4e0b089813d5647e1bdcd706ac651e67847c8a...,f22c891361a2bcc2ca93c23104632357
3,2.336367,NaN,54f84a1d4e0b089813d5647e1bdcd706ac651e67847c8a...,f22c891361a2bcc2ca93c23104632357
4,2.336582,NaN,4ef84a1d4e0b08981379de027e2b0b60000b537d248e4f...,f22c891361a2bcc2ca93c23104632357
...,...,...,...,...
96099,32.698808,NaN,4bcff6034eac4d18657bf4ccca1dc8cfaa1d955ce620ce...,73b8a303944602ec5f1d5c58efcf728b
96100,32.698811,NaN,4bcff6034eac4d18657bf4ccca1dc8cfaa1d955ce620ce...,73b8a303944602ec5f1d5c58efcf728b
96101,32.698901,NaN,43cff6034eac4d1865a84253c71dc5a0576964e70e24a4...,73b8a303944602ec5f1d5c58efcf728b
96102,32.698905,NaN,43cff6034eac4d1865a84253c71dc5a0576964e70e24a4...,73b8a303944602ec5f1d5c58efcf728b


## Ahora nos toca procesar los flujos de cada dataframe, viendo si son válidos primero, es decir, si llegan al mínimo establecido de paquetes. En caso afirmativo se divididen en semiflujos, que son los que realmente van a alimentar a las redes neuronales que entrenemos.
## Los semiflujos que se vayan generando se guardarán en una lista, y su correspondiente etiqueta se guardará en otra lista distinta. Esto generará una lista de semiflujos y otra de etiquetas

In [11]:
start_time = time.time()

## Un diccionario que servirá para saber cuántos semiflujos se han generado de cada tipo
Semiflujos_diccionario_contenido = {}

## La lista global que contendrá todos los semiflujos, posteriormente se transformará a numpy array para 
## un uso óptimo de memoria
lista_semiflujos = list()

## Recorremos el diccionario de dataframes, categoría a categoría
for key in DF_diccionario:
    
    ## Vemos que tamaño tiene la lista de semiflujos actualmente, servirá para saber cuántos semiflujos
    ## hemos añadido de la categoría en construcción
    tam_lista = len(lista_semiflujos)
     
    ## Vemos cuantos dataframes contiene esta categoría
    num_dfs = len(DF_diccionario[key])
    print("\n\n\nComenzando la lista de dataframes de tipo "+key+"\n")
    
    ## Recorremos todos los dataframes de la categoría que represente la variable 'key'
    for cont,DF in enumerate(DF_diccionario[key]):    
        
        ## Este es un proceso largo así que es necesario un buen feedback visual para saber cómo va progresando el programa
        print("\n\n----Comenzando el DF numero "+str(cont+1)+" de "+str(num_dfs)+" de tipo "+key)
        
        ## Vamos a obtener una lista de flujos del dataframe con el que estamos operando
        lista_flujos = obtenerFlujos(DF, min_paquetes_flujo)    
        print("----Numero de flujos obtenidos en el DF igual a "+str(len(lista_flujos))+"\n")
        
        ## Función que dada una lista de flujos, añade los semiflujos que encuentre a la lista de semiflujos que recibe como argumento
        obtenerSemiflujos(lista_flujos, lista_semiflujos, token_separador, num_paquetes_semiflujo, tiempo_maximo_semiflujo,longitud_raw,num_hex_token)
        
    ## Vemos que tamaño tiene la lista de semiflujos después de haber añadido los nuevos DFs
    tam_lista_nuevo = len(lista_semiflujos)
    
    total_semiflujos_agregados = tam_lista_nuevo - tam_lista
    
    ## Guardamos en el diccionario de contenido cuántos semiflujos hemos generado de esta categoría
    Semiflujos_diccionario_contenido[key] = total_semiflujos_agregados

    
## A continuación se deja la lista como un np array tridimensional, donde la primera dimensión es el número de semiflujos,
## la segunda es los paquetes que compone dicho semiflujo, y la última es los tokens en los que se divide el paquete
lista_semiflujos = dstack(lista_semiflujos)
lista_semiflujos = np.rollaxis(lista_semiflujos,-1) ## Necesario pues las dimensiones no estaban en el orden correcto
    
calcular_tiempo(start_time)




Comenzando la lista de dataframes de tipo YouTube



----Comenzando el DF numero 1 de 4 de tipo YouTube
----Numero de flujos obtenidos en el DF igual a 4

--------Comenzando el flujo numero 1 de 4
--------Comenzando el flujo numero 2 de 4
--------Comenzando el flujo numero 3 de 4
--------Comenzando el flujo numero 4 de 4


----Comenzando el DF numero 2 de 4 de tipo YouTube
----Numero de flujos obtenidos en el DF igual a 1

--------Comenzando el flujo numero 1 de 1


----Comenzando el DF numero 3 de 4 de tipo YouTube
----Numero de flujos obtenidos en el DF igual a 2

--------Comenzando el flujo numero 1 de 2
--------Comenzando el flujo numero 2 de 2


----Comenzando el DF numero 4 de 4 de tipo YouTube
----Numero de flujos obtenidos en el DF igual a 10

--------Comenzando el flujo numero 1 de 10
--------Comenzando el flujo numero 2 de 10
--------Comenzando el flujo numero 3 de 10
--------Comenzando el flujo numero 4 de 10
--------Comenzando el flujo numero 5 de 10
--------Comenzando e

--------Comenzando el flujo numero 88 de 135
--------Comenzando el flujo numero 89 de 135
--------Comenzando el flujo numero 90 de 135
--------Comenzando el flujo numero 91 de 135
--------Comenzando el flujo numero 92 de 135
--------Comenzando el flujo numero 93 de 135
--------Comenzando el flujo numero 94 de 135
--------Comenzando el flujo numero 95 de 135
--------Comenzando el flujo numero 96 de 135
--------Comenzando el flujo numero 97 de 135
--------Comenzando el flujo numero 98 de 135
--------Comenzando el flujo numero 99 de 135
--------Comenzando el flujo numero 100 de 135
--------Comenzando el flujo numero 101 de 135
--------Comenzando el flujo numero 102 de 135
--------Comenzando el flujo numero 103 de 135
--------Comenzando el flujo numero 104 de 135
--------Comenzando el flujo numero 105 de 135
--------Comenzando el flujo numero 106 de 135
--------Comenzando el flujo numero 107 de 135
--------Comenzando el flujo numero 108 de 135
--------Comenzando el flujo numero 109 de 135


--------Comenzando el flujo numero 52 de 126
--------Comenzando el flujo numero 53 de 126
--------Comenzando el flujo numero 54 de 126
--------Comenzando el flujo numero 55 de 126
--------Comenzando el flujo numero 56 de 126
--------Comenzando el flujo numero 57 de 126
--------Comenzando el flujo numero 58 de 126
--------Comenzando el flujo numero 59 de 126
--------Comenzando el flujo numero 60 de 126
--------Comenzando el flujo numero 61 de 126
--------Comenzando el flujo numero 62 de 126
--------Comenzando el flujo numero 63 de 126
--------Comenzando el flujo numero 64 de 126
--------Comenzando el flujo numero 65 de 126
--------Comenzando el flujo numero 66 de 126
--------Comenzando el flujo numero 67 de 126
--------Comenzando el flujo numero 68 de 126
--------Comenzando el flujo numero 69 de 126
--------Comenzando el flujo numero 70 de 126
--------Comenzando el flujo numero 71 de 126
--------Comenzando el flujo numero 72 de 126
--------Comenzando el flujo numero 73 de 126
--------Co

--------Comenzando el flujo numero 51 de 91
--------Comenzando el flujo numero 52 de 91
--------Comenzando el flujo numero 53 de 91
--------Comenzando el flujo numero 54 de 91
--------Comenzando el flujo numero 55 de 91
--------Comenzando el flujo numero 56 de 91
--------Comenzando el flujo numero 57 de 91
--------Comenzando el flujo numero 58 de 91
--------Comenzando el flujo numero 59 de 91
--------Comenzando el flujo numero 60 de 91
--------Comenzando el flujo numero 61 de 91
--------Comenzando el flujo numero 62 de 91
--------Comenzando el flujo numero 63 de 91
--------Comenzando el flujo numero 64 de 91
--------Comenzando el flujo numero 65 de 91
--------Comenzando el flujo numero 66 de 91
--------Comenzando el flujo numero 67 de 91
--------Comenzando el flujo numero 68 de 91
--------Comenzando el flujo numero 69 de 91
--------Comenzando el flujo numero 70 de 91
--------Comenzando el flujo numero 71 de 91
--------Comenzando el flujo numero 72 de 91
--------Comenzando el flujo nume

--------Comenzando el flujo numero 28 de 66
--------Comenzando el flujo numero 29 de 66
--------Comenzando el flujo numero 30 de 66
--------Comenzando el flujo numero 31 de 66
--------Comenzando el flujo numero 32 de 66
--------Comenzando el flujo numero 33 de 66
--------Comenzando el flujo numero 34 de 66
--------Comenzando el flujo numero 35 de 66
--------Comenzando el flujo numero 36 de 66
--------Comenzando el flujo numero 37 de 66
--------Comenzando el flujo numero 38 de 66
--------Comenzando el flujo numero 39 de 66
--------Comenzando el flujo numero 40 de 66
--------Comenzando el flujo numero 41 de 66
--------Comenzando el flujo numero 42 de 66
--------Comenzando el flujo numero 43 de 66
--------Comenzando el flujo numero 44 de 66
--------Comenzando el flujo numero 45 de 66
--------Comenzando el flujo numero 46 de 66
--------Comenzando el flujo numero 47 de 66
--------Comenzando el flujo numero 48 de 66
--------Comenzando el flujo numero 49 de 66
--------Comenzando el flujo nume

--------Comenzando el flujo numero 123 de 149
--------Comenzando el flujo numero 124 de 149
--------Comenzando el flujo numero 125 de 149
--------Comenzando el flujo numero 126 de 149
--------Comenzando el flujo numero 127 de 149
--------Comenzando el flujo numero 128 de 149
--------Comenzando el flujo numero 129 de 149
--------Comenzando el flujo numero 130 de 149
--------Comenzando el flujo numero 131 de 149
--------Comenzando el flujo numero 132 de 149
--------Comenzando el flujo numero 133 de 149
--------Comenzando el flujo numero 134 de 149
--------Comenzando el flujo numero 135 de 149
--------Comenzando el flujo numero 136 de 149
--------Comenzando el flujo numero 137 de 149
--------Comenzando el flujo numero 138 de 149
--------Comenzando el flujo numero 139 de 149
--------Comenzando el flujo numero 140 de 149
--------Comenzando el flujo numero 141 de 149
--------Comenzando el flujo numero 142 de 149
--------Comenzando el flujo numero 143 de 149
--------Comenzando el flujo numero

1

### Vamos a comprobar qué resultados hemos obtenido después de generar los semiflujos

In [12]:
total_semiflujos_procesados = 0

for key in Semiflujos_diccionario_contenido:
    num_semiflujos = Semiflujos_diccionario_contenido[key]
    print("\nHemos obtenido una lista de "+str(num_semiflujos)+" semiflujos del tipo "+key)
    total_semiflujos_procesados+=num_semiflujos
    
print("\nEl total de semiflujos registrado en el diccionario de contenido es "+ str(total_semiflujos_procesados))

print("El total de semiflujos del array tridimensional es "+str(len(lista_semiflujos)))

print("Las dimensiones del array tridimensional son: "+str(lista_semiflujos.shape))


Hemos obtenido una lista de 18058 semiflujos del tipo YouTube

Hemos obtenido una lista de 22688 semiflujos del tipo Spotify

Hemos obtenido una lista de 25488 semiflujos del tipo Discord

El total de semiflujos registrado en el diccionario de contenido es 66234
El total de semiflujos del array tridimensional es 66234
Las dimensiones del array tridimensional son: (66234, 20, 201)


## Debido a la naturaleza de los datos, es posible que se generen semiflujos repetidos, ya que las conexiones de red siguen ciertos patrones que siempre se van a repetir, como la conexión inicial, es por esto que vamos a comprobar cuantos semiflujos idénticos existen, para eliminar los repetidos y no sesgar el modelo

In [13]:
## Calculamos el número de semiflujos únicos
comprobacion_datos_unicos = np.unique(lista_semiflujos,axis=0)
print("De un total de "+str(lista_semiflujos.shape[0])+" de semiflujos, realmente hay "+str(comprobacion_datos_unicos.shape[0])+" unicos")
num_repetidos = lista_semiflujos.shape[0] - comprobacion_datos_unicos.shape[0]
porcentaje_repetidos = round(num_repetidos/lista_semiflujos.shape[0]*100 ,2)
print("\nEsto supone un "+str(porcentaje_repetidos)+"% de semiflujos repetidos")

De un total de 66234 de semiflujos, realmente hay 66233 unicos

Esto supone un 0.0% de semiflujos repetidos


### Una vez hemos identificado cuántos repetidos existen, procedemos a eliminarlos

In [14]:
## La función 'desechar_semiflujos_repetidos' devuelve el array que recibe como argumento pero sin semiflujos 
## repetidos y también su nuevo array de etiquetas correspondiente

semiflujos_unicos,lista_arrays_etiqueta_categorical = desechar_semiflujos_repetidos(lista_semiflujos, Semiflujos_diccionario_contenido, num_clases_existentes)


Hay 1 de semiflujos repetidos de 18058 totales en la categoria YouTube
Esto representa el 0.01% de la categoría
Hay 18057 flujos unicos realmente

Hay 0 de semiflujos repetidos de 22688 totales en la categoria Spotify
Esto representa el 0.0% de la categoría
Hay 22688 flujos unicos realmente

Hay 0 de semiflujos repetidos de 25488 totales en la categoria Discord
Esto representa el 0.0% de la categoría
Hay 25488 flujos unicos realmente

Vamos a ver como se quedan las dimensiones de los datos después de quitar los repetidos:
Array semiflujos: (66233, 20, 201)
Array etiquetas: (66233, 3)
Diccionario de contenido: {'YouTube': 18057, 'Spotify': 22688, 'Discord': 25488}


In [15]:
## Vamos a comprobar como el nuevo array que nos ha devuelto la función anterior ya no contiene datos repetidos
comprobacion_datos_unicos = np.unique(semiflujos_unicos,axis=0)
print("Check final de que no hay datos repetidos\n")

print("Dimensionalidad del array de semiflujos unico obtenido en el paso anterior: "+str(semiflujos_unicos.shape))
print("Dimensionalidad del array resultante de aplicar 'np.unique' al array de semiflujos unico: "+str(comprobacion_datos_unicos.shape))
## (Deberían coincidir si todo ha ido bien)

Check final de que no hay datos repetidos

Dimensionalidad del array de semiflujos unico obtenido en el paso anterior: (66233, 20, 201)
Dimensionalidad del array resultante de aplicar 'np.unique' al array de semiflujos unico: (66233, 20, 201)


### Imprimimos un paquete de un semiflujo aleatorio del array para hacernos una idea de cómo se han quedado

In [16]:
semiflujos_unicos[99][0]

array([16428, 33039, 64368, 59860, 28774, 47070, 61307, 36750, 36711,
        7823, 41895, 18954, 41753, 14629, 45811, 54676, 47550,  3373,
        5759, 58949,  1499, 13517, 50053, 65099, 24537, 18968, 56858,
       29392, 30845, 56116, 63454,  6783, 59386,  9762, 48370,  2651,
       55642, 33641, 14773, 32732, 54062, 54027, 20349,  9429,   655,
       12133, 11400, 25666, 62603, 18157, 23737, 52638, 49170, 49975,
       17145, 33036, 52935, 45460, 30188, 11413, 54393, 21747,   932,
        4973, 57833,  9105, 11384, 59148, 46528, 63569, 52548, 23364,
       35700,  8022, 52436, 53179, 63282, 59249, 65379, 35066, 50724,
       19780, 14710,  1267, 51317, 11196,  8134, 37225, 49980, 52793,
        5866, 60035, 24804, 40478,  7152, 17363, 43291, 43686, 41825,
       32546, 21814, 13109, 22075, 42675, 61157, 25847,  5515, 11935,
       64840, 49302, 56087, 37166, 40435,  1946, 27005, 19984, 55515,
       61548,  2258, 36122, 33373, 60400, 40851, 34476, 52303, 26496,
        2785,  5334,

## Ya tenemos la lista de semiflujos, este es un proceso largo y que realmente solo deberíamos realizar una vez para un conjunto de hiperparámetros dados
## Usando dicha lista entrenaremos el modelo de IA, así que vamos a guardarla de forma persistente en el equipo, para poder recuperarla fácil y rápidamente cuando la vayamos a usar

In [17]:
## Vamos a guardar en la carpeta correspondiente la lista de semiflujos, sus etiquetas, el diccionario de contenido
## y el archivo de configuración utilizado
## Para organizar todos estos archivos, vamos a nombrarlos con el hash que hemos calculado al principio del
## preprocesamiento, ya que lo identifica de manera inequívoca

## Llamamos a la función que nos guarda todo en el disco
guardar_semiflujos(Semiflujos_diccionario_contenido, semiflujos_unicos, lista_arrays_etiqueta_categorical,pathRelativoSemiflujos,hashSemiflujos)

## Esta función actualiza un CSV con los datos de todos los semiflujos guardados hasta el momento
## El CSV se llama "caracteristicas_semiflujos.CSV" y se guarda en la misma carpeta donde están todos los semiflujos
evaluar_semiflujos_guardados(pathRelativoSemiflujos)

Directorio para los semiflujos 060eaab55e1e0dd2d41fe88ed69ebb5b ya existente! Contenido actualizado


### Obtenemos las características de uso del PC durante el preprocesamiento

In [18]:
CPU,RAM,picoRAM,nombre_GPU,VRAM,picoVRAM = obtener_especificaciones_pc()


PID del entrenamiento = 219

Características del PC donde se ha ejecutado el entrenamiento:
- CPU = AMD Ryzen 7 5700X 8-Core Processor
- RAM = 64GB
- Pico de uso de RAM = 21.6GB
- GPU = NVIDIA GeForce RTX 3090
- VRAM = 24GB
- Pico de uso de VRAM = 0.0GB


2023-07-08 15:52:38.734277: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1635] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22053 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:26:00.0, compute capability: 8.6


## ------------------------------------- 🎉 FIN DEL NOTEBOOK 🎉 -------------------------------------